# Agentic Research Assistant with RAG

An end-to-end agent workflow for planning, retrieval, grounded synthesis, citation checking, and trace export.

## 1. Setup

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import pandas as pd

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

sources = pd.read_csv(DATA_DIR / 'sample_sources.csv')
questions = pd.read_csv(DATA_DIR / 'sample_questions.csv')
sources

## 2. Agent State Schema

In [ ]:
def make_state(question_row):
    return {
        'question_id': question_row.question_id,
        'question': question_row.question,
        'expected_sources': str(question_row.expected_sources).split(';'),
        'plan': [],
        'retrieved_sources': [],
        'answer': None,
        'citations': [],
        'checks': {},
    }

states = [make_state(row) for row in questions.itertuples()]
states[0]

## 3. Planner Node

In [ ]:
def plan_research(state):
    text = state['question'].lower()
    plan = ['retrieve_evidence', 'synthesize_answer', 'check_citations']
    if any(term in text for term in ['evaluate', 'should', 'when']):
        plan.insert(1, 'inspect_risk_or_quality')
    state['plan'] = plan
    return state

states = [plan_research(state) for state in states]
pd.DataFrame([{'question_id': s['question_id'], 'plan': s['plan']} for s in states])

## 4. Retriever Node

In [ ]:
def tokenize(text):
    return [t.strip('.,?!;:').lower() for t in str(text).split() if t.strip('.,?!;:')]

def cosine_score(query, document):
    q = Counter(tokenize(query))
    d = Counter(tokenize(document))
    numerator = sum(q[token] * d.get(token, 0) for token in q)
    q_norm = math.sqrt(sum(v*v for v in q.values()))
    d_norm = math.sqrt(sum(v*v for v in d.values()))
    return 0 if q_norm == 0 or d_norm == 0 else numerator / (q_norm * d_norm)

def retrieve(state, top_k=2):
    rows = []
    for row in sources.itertuples():
        text = f"{row.title} {row.domain} {row.text}"
        rows.append((row.source_id, cosine_score(state['question'], text)))
    state['retrieved_sources'] = sorted(rows, key=lambda item: item[1], reverse=True)[:top_k]
    return state

states = [retrieve(state) for state in states]
pd.DataFrame([{'question_id': s['question_id'], 'retrieved_sources': s['retrieved_sources']} for s in states])

## 5. Synthesizer Node

In [ ]:
source_lookup = sources.set_index('source_id').to_dict(orient='index')

def synthesize(state):
    top_ids = [source_id for source_id, score in state['retrieved_sources']]
    evidence = [source_lookup[source_id]['text'] for source_id in top_ids]
    state['answer'] = ' '.join(evidence)
    state['citations'] = top_ids
    return state

states = [synthesize(state) for state in states]
pd.DataFrame([{'question_id': s['question_id'], 'answer': s['answer'], 'citations': s['citations']} for s in states])

## 6. Citation Checker Node

In [ ]:
def check_citations(state):
    expected = set(state['expected_sources'])
    cited = set(state['citations'])
    state['checks']['source_recall'] = len(expected & cited) / max(len(expected), 1)
    state['checks']['has_citation'] = bool(cited)
    state['checks']['trace_complete'] = all([state['plan'], state['retrieved_sources'], state['answer'], state['citations']])
    return state

states = [check_citations(state) for state in states]
pd.DataFrame([{'question_id': s['question_id'], **s['checks']} for s in states])

## 7. Export Agent Trace

In [ ]:
trace_rows = []
for state in states:
    trace_rows.append({
        'question_id': state['question_id'],
        'question': state['question'],
        'plan': ' > '.join(state['plan']),
        'retrieved_sources': ';'.join([source for source, _ in state['retrieved_sources']]),
        'citations': ';'.join(state['citations']),
        'answer': state['answer'],
        **state['checks'],
    })
trace = pd.DataFrame(trace_rows)
trace.to_csv(OUTPUT_DIR / 'research_agent_trace.csv', index=False)
trace

## 8. Evaluation Summary

In [ ]:
summary = {
    'mean_source_recall': trace['source_recall'].mean(),
    'citation_rate': trace['has_citation'].mean(),
    'trace_completeness': trace['trace_complete'].mean(),
}
summary